In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. Instala as dependências menores que o PandasAI precisa
!pip install astor faker openai duckdb pydantic

# 2. Instala o PandasAI ignorando a exigência de fazer downgrade do Pandas
!pip install pandasai --no-deps

#### **POLARS**

https://pola.rs/

In [ ]:
import polars as pl

In [ ]:
# 1. Criação de DataFrame
data = {
    'idade': [25, 32, 47, 51, 62],
    'imc': [22.4, 28.9, 30.2, 26.4, 24.8],
    'colesterol': [190, 230, 210, 180, 200]
}
df = pl.DataFrame(data)
print("DataFrame Original:")
print(df)

In [ ]:
# 2. Seleção de Colunas
df_selecionado = df.select(['idade', 'imc'])
print("\nColunas Selecionadas (idade, imc):")
print(df_selecionado)

In [ ]:
# 3. Filtragem de Dados
df_filtrado = df.filter(pl.col('idade') > 30)
print("\nDados Filtrados (idade > 30):")
print(df_filtrado)

In [ ]:
# 4. Adição de Colunas
df = df.with_columns((pl.col('imc') / pl.col('idade')).alias('imc_por_idade'))
print("\nAdição de Coluna (imc_por_idade):")
print(df)

In [ ]:
# 5. Agregações
df_agrupado = df.group_by('idade').agg(pl.mean('imc').alias('media_imc'))
print("\nAgregação (média do IMC por idade):")
print(df_agrupado)

In [ ]:
# 6. Junções
data2 = {
    'idade': [25, 32, 47, 51, 62],
    'pressao_arterial': [120, 130, 140, 150, 160]
}
df2 = pl.DataFrame(data2)
df_juncao = df.join(df2, on='idade', how='inner')
print("\nJunção de DataFrames:")
print(df_juncao)

In [ ]:
# 7. Ordenação
df_ordenado = df.sort('imc', descending=True)
print("\nOrdenação (IMC em ordem decrescente):")
print(df_ordenado)

##### **Comparação de Performance entre Pandas e Polars**

Para demonstrar a diferença de performance entre Pandas e Polars, vamos usar um conjunto de dados grande. O conjunto de dados escolhido para este exemplo é o dataset de transações de táxi de Nova York, que é bastante volumoso e pode ser baixado de fontes públicas.

**Passos do Caso de Estudo**

**Download e Preparação dos Dados**

1. Baixar um grande volume de dados do dataset de transações de táxi de Nova York.
2. Processar e salvar os dados para uso no estudo.

**Operações de Manipulação de Dados**

1. Realizar operações de leitura, filtragem, agregação e ordenação com Pandas e Polars.
2. Medir e comparar o tempo de execução das operações.
3. Download e Preparação dos Dados
> Utilizaremos o dataset "Yellow Taxi Trip Records" de janeiro de 2023 para nosso caso de estudo. Este dataset pode ser encontrado [aqui](https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet).

In [ ]:
import time
import pandas as pd
import polars as pl

In [ ]:
# Download do dataset de janeiro de 2023
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
df = pd.read_parquet(url)

# Salvando um arquivo CSV para facilitar a leitura com Polars
df.to_csv('yellow_tripdata_2023-01.csv', index=False)

**PANDAS**

In [ ]:
# Leitura dos dados com especificação de tipos de dados e low_memory=False
start_time = time.time()
df_pandas = pd.read_csv('yellow_tripdata_2023-01.csv', dtype={'PULocationID': 'int32', 'DOLocationID': 'int32', 'passenger_count': 'float32', 'trip_distance': 'float32', 'fare_amount': 'float32', 'total_amount': 'float32'}, low_memory=False)
print(f"Pandas: Leitura dos dados levou {time.time() - start_time:.2f} segundos")

# Filtragem de dados
start_time = time.time()
df_pandas_filtered = df_pandas[df_pandas['trip_distance'] > 2]
print(f"Pandas: Filtragem dos dados levou {time.time() - start_time:.2f} segundos")

# Agregação dos dados
start_time = time.time()
df_pandas_agg = df_pandas_filtered.groupby('PULocationID').agg({'trip_distance': 'mean', 'total_amount': 'sum'})
print(f"Pandas: Agregação dos dados levou {time.time() - start_time:.2f} segundos")

# Ordenação dos dados
start_time = time.time()
df_pandas_sorted = df_pandas_agg.sort_values(by='total_amount', ascending=False)
print(f"Pandas: Ordenação dos dados levou {time.time() - start_time:.2f} segundos")

**POLARS**

In [ ]:
# Leitura dos dados
start_time = time.time()
df_polars = pl.read_csv('yellow_tripdata_2023-01.csv')
print(f"Polars: Leitura dos dados levou {time.time() - start_time:.2f} segundos")

# Filtragem de dados
start_time = time.time()
df_polars_filtered = df_polars.filter(pl.col('trip_distance') > 2)
print(f"Polars: Filtragem dos dados levou {time.time() - start_time:.2f} segundos")

# Agregação dos dados
start_time = time.time()
df_polars_agg = df_polars_filtered.group_by('PULocationID').agg([
    pl.mean('trip_distance').alias('trip_distance_mean'),
    pl.sum('total_amount').alias('total_amount_sum')
])
print(f"Polars: Agregação dos dados levou {time.time() - start_time:.2f} segundos")

# Ordenação dos dados
start_time = time.time()
df_polars_sorted = df_polars_agg.sort('total_amount_sum', descending=True)
print(f"Polars: Ordenação dos dados levou {time.time() - start_time:.2f} segundos")

Ao comparar os tempos de execução de cada operação entre Pandas e Polars, esperamos observar que Polars realiza essas operações de maneira significativamente mais rápida, especialmente em datasets grandes, devido à sua eficiência e paralelismo otimizado.

#### **PANDAS AI**

https://pandas-ai.com/

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import userdata
from pandasai import Agent
from pandasai.llm import GoogleGemini
from sklearn.linear_model import LinearRegression
import google.generativeai as genai

In [ ]:
# Pegando a chave com o nome correto que salvamos nos Secrets
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

In [ ]:
llm = GoogleGemini(api_key=os.environ['GEMINI_API_KEY'])
llm.google_gemini = genai.GenerativeModel("gemini-2.5-flash")

In [ ]:
# Criar DataFrame de exemplo
data = {
    'idade': [25, 32, 47, 51, 62, 38, 29, 45, 56, 40],
    'imc': [22.4, 28.9, 30.2, 26.4, 24.8, 27.1, 23.4, 29.5, 31.1, 25.6],
    'colesterol': [190, 230, 210, 180, 200, 220, 195, 205, 215, 200]
}
df = pd.DataFrame(data)

In [ ]:
agent = Agent(df, config={"llm": llm})

In [ ]:
response = agent.chat("Qual valor de colesterol previsto para um paciente de 60 anos e imc de 35?")
print(response)

In [ ]:
response = agent.chat("Qual valor de colesterol previsto para um paciente de 47 anos e imc de 30.2?")
print(response)

In [ ]:
response = agent.chat("Qual seria a idade aproximada de um paciente com IMC de 30 e colesterol 215?")
print(response)

In [ ]:
import re

# Função para garantir que a resposta é numérica (Filtro Anti-Conversa)
def get_numeric_response(prompt):
    # Forçando a IA a ser direta
    prompt_estrito = prompt + " Responda APENAS com o valor numérico (ex: 202.59), sem nenhuma palavra adicional."

    response = str(agent.chat(prompt_estrito))

    try:
        # Troca vírgula por ponto
        response_limpa = response.replace(',', '.')

        # Puxa qualquer número (inteiro ou decimal) que estiver no meio do texto
        numeros = re.findall(r"[-+]?\d*\.\d+|\d+", response_limpa)

        if numeros:
            return float(numeros[0])
        else:
            return float(response_limpa)

    except Exception as e:
        raise ValueError(f"Erro ao extrair número. Resposta da IA: '{response}'")

# Criar novos dados para previsão
new_data = pd.DataFrame({
    'idade': [60, 55, 50],
    'imc': [35, 28, 26]
})

# Fazer previsões
new_data['colesterol'] = [get_numeric_response(f"Qual valor de colesterol previsto para um paciente de {idade} anos e imc de {imc}?") for idade, imc in zip(new_data['idade'], new_data['imc'])]

# Unir os DataFrames
df_updated = pd.concat([df, new_data], ignore_index=True)

print(df_updated)

In [ ]:
print(df_updated)

In [ ]:
# Treinamento do modelo de regressão linear com sklearn para plotagem
X = df[['idade', 'imc']]
y = df['colesterol']
model = LinearRegression()
model.fit(X, y)

# Previsão para a linha de regressão
X_full = df_updated[['idade', 'imc']]
y_pred_full = model.predict(X_full)

In [ ]:
# Plotando os dados originais e previstos em um gráfico 3D
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Dados originais
ax.scatter(df['idade'], df['imc'], df['colesterol'], color='blue', label='Dados Originais')

# Novos dados previstos
ax.scatter(new_data['idade'], new_data['imc'], new_data['colesterol'], color='red', label='Novos Dados Previstos')

# Linha de regressão
ax.plot(df_updated['idade'], df_updated['imc'], y_pred_full, color='green', linestyle='--', label='Linha de Regressão')

ax.set_xlabel('Idade')
ax.set_ylabel('IMC')
ax.set_zlabel('Colesterol')
ax.set_title('Correlação entre Idade, IMC e Índice de Colesterol')
ax.legend()

plt.show()

In [ ]:
# Calcular a matriz de correlação
correlation_matrix = df_updated.corr()

# Plotar a matriz de correlação
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Matriz de Correlação dos Dados')
plt.show()

In [ ]:
from pandasai import SmartDatalake
from pandasai.responses.streamlit_response import StreamlitResponse

agent = SmartDatalake(
    [df_updated],
    config={
      "llm": llm,
      "verbose": True,
      "response_parser": StreamlitResponse
    },
)

agent.chat("Mostre em um gráfico a correlação entre as variáveis imc, idade e colesterol")

In [ ]:
from IPython.display import Image

# Mostra a imagem salva no caminho que o PandasAI devolveu
Image('/content/exports/charts/temp_chart.png')

### **EXERCÍCIO PRÁTICO**

Você recebeu um conjunto de dados contendo informações sobre a idade, anos de experiência e salário de um grupo de profissionais. O objetivo é treinar um modelo de regressão linear para prever o salário com base na idade e nos anos de experiência, e visualizar a correlação entre essas variáveis.

```
import pandas as pd

# Criar DataFrame de exemplo
data = {
    'idade': [25, 32, 47, 51, 62, 38, 29, 45, 56, 40, 35, 31, 28, 50, 48],
    'anos_experiencia': [2, 5, 20, 25, 30, 10, 4, 18, 28, 12, 7, 5, 3, 22, 19],
    'salario': [35000, 48000, 80000, 90000, 120000, 55000, 42000, 75000, 110000, 60000, 52000, 47000, 40000, 85000, 82000]
}
df = pd.DataFrame(data)
```

**Passos do Exercício**

1. Treinamento do Modelo:

+ Utilize PandasAI para treinar um modelo de regressão linear que preveja o salário com base na idade e anos de experiência.

2. Previsão para Novos Dados:

+ Crie um novo conjunto de dados com valores de idade e anos de experiência que não existem no conjunto de dados original e use PandasAI para prever os salários para esses novos dados.

3. Visualização dos Resultados:

+ Plote a matriz de correlação entre idade, anos de experiência e salário.
+ Crie um gráfico de dispersão 3D que mostre a relação entre idade, anos de experiência e salário, diferenciando em cores os novos dados previstos dos dados originais.

# RESOLUÇÃO


In [ ]:
# 1. Preparação dos Dados Originais do Exercício
data_exercicio = {
    'idade': [25, 32, 47, 51, 62, 38, 29, 45, 56, 40, 35, 31, 28, 50, 48],
    'anos_experiencia': [2, 5, 20, 25, 30, 10, 4, 18, 28, 12, 7, 5, 3, 22, 19],
    'salario': [35000, 48000, 80000, 90000, 120000, 55000, 42000, 75000, 110000, 60000, 52000, 47000, 40000, 85000, 82000]
}
df_exercicio = pd.DataFrame(data_exercicio)

# Instanciamos o Agente para esse novo DataFrame (o 'llm' do Gemini já está na memória!)
agent_exercicio = Agent(df_exercicio, config={"llm": llm})

# Função com filtro regex adaptada para este agente
def get_numeric_response_exercicio(prompt):
    prompt_estrito = prompt + " Responda APENAS com o valor numérico em formato float, sem nenhuma palavra ou símbolo."
    response = str(agent_exercicio.chat(prompt_estrito))
    try:
        response_limpa = response.replace(',', '.')
        numeros = re.findall(r"[-+]?\d*\.\d+|\d+", response_limpa)
        if numeros:
            return float(numeros[0])
        else:
            return float(response_limpa)
    except Exception as e:
        raise ValueError(f"Erro ao extrair número. Resposta da IA: '{response}'")

# ==========================================
# PASSO 1 E 2: Previsão para Novos Dados
# ==========================================

# Criando 3 perfis novos que não existem na tabela original
new_data = pd.DataFrame({
    'idade': [27, 43, 58],
    'anos_experiencia': [4, 16, 27]
})

print("Calculando previsões de salário com IA (isso pode levar alguns segundos)...")
new_data['salario'] = [get_numeric_response_exercicio(f"Treine um modelo de regressão linear. Qual o salário previsto para um profissional de {idade} anos de idade e {exp} anos de experiência?") for idade, exp in zip(new_data['idade'], new_data['anos_experiencia'])]

# Unindo tudo em um DataFrame final
df_final = pd.concat([df_exercicio, new_data], ignore_index=True)
print("\n--- DataFrame Atualizado (Últimas linhas são as previsões) ---")
print(df_final.tail())

# ==========================================
# PASSO 3: Visualização dos Resultados
# ==========================================

# A. Matriz de Correlação
plt.figure(figsize=(8, 6))
correlation_matrix = df_final.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Matriz de Correlação: Idade, Experiência e Salário')
plt.show()

# Treinamento com Scikit-Learn apenas para traçar a linha matemática no gráfico 3D
X = df_exercicio[['idade', 'anos_experiencia']]
y = df_exercicio['salario']
model_plot = LinearRegression()
model_plot.fit(X, y)
y_pred_full = model_plot.predict(df_final[['idade', 'anos_experiencia']])

# B. Gráfico de Dispersão 3D
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plotar dados originais (Bolhas azuis)
ax.scatter(df_exercicio['idade'], df_exercicio['anos_experiencia'], df_exercicio['salario'], color='blue', label='Dados Originais', s=50)

# Plotar novos dados previstos (Triângulos vermelhos)
ax.scatter(new_data['idade'], new_data['anos_experiencia'], new_data['salario'], color='red', label='Previsões PandasAI', marker='^', s=80)

# Traçar a linha de regressão
ax.plot(df_final['idade'], df_final['anos_experiencia'], y_pred_full, color='green', linestyle='--', label='Linha de Regressão')

ax.set_xlabel('Idade (Anos)')
ax.set_ylabel('Experiência (Anos)')
ax.set_zlabel('Salário (R$)')
ax.set_title('Previsão de Salários com Regressão Linear')
ax.legend()

plt.show()